# 01. The task and the box

The state-tracking task from the paper (arXiv:2602.14872). States and actions are numbers mod $d=96$. A problem of length $L$ gives a start state $y_0$ and $L$ instructions; the chain is

$$y_{k+1} = (y_k + a_k) \bmod 96,$$

and only the final state $y_L$ is graded. The catch: instructions are not in order. Each one carries a **tag**, and each answer slot carries the **partner tag** under a secret codebook. To find slot $k$'s action, the model has to learn the codebook.

In [1]:
import sys; sys.path.append('..')   # the code lives in the repo root
import torch, torch.nn.functional as F
from data import d, make_codebook, get_batch, show

codebook = make_codebook(seed=0)
batch = get_batch(torch.tensor([3, 15]), codebook)
show(batch, b=0)

L = 3
prompt (tag, action): (t1, +40)  (t57, +46)  (t39, +78)
answer (tag, state) : (t46, 85)  (t12, 29)  (t5, 75)  (t17, 57)
the model is given the start state 85 and the answer tags; it is graded on 57


## Solving one problem by hand

This is the lookup the model has to learn: stand on an answer tag, find its partner prompt tag, apply that instruction's action.

In [2]:
partner = torch.argsort(codebook)           # partner[answer tag] = prompt tag
p, g, a, y = (batch[k][0].tolist() for k in ('p_tag', 'act', 'a_tag', 'y'))
state = y[0]
for k in range(batch['L'][0].item()):
    pos = p.index(partner[a[k]].item())        # which instruction carries the partner tag?
    print(f'slot {k}: t{a[k]} -> partner t{p[pos]} -> action +{g[pos]} -> ({state} + {g[pos]}) % {d} = {(state + g[pos]) % d}')
    state = (state + g[pos]) % d
print('final', state, '| true answer', batch['y_final'][0].item())

slot 0: t46 -> partner t1 -> action +40 -> (85 + 40) % 96 = 29
slot 1: t12 -> partner t57 -> action +46 -> (29 + 46) % 96 = 75
slot 2: t5 -> partner t39 -> action +78 -> (75 + 78) % 96 = 57
final 57 | true answer 57


## The box: one step, learned once, then frozen

As in the paper, the model already knows how to apply one action. An MLP is trained on all $96\times 96$ (state, action) pairs and then frozen. `load_box()` trains it the first time (about a minute) and saves it to `box.pt`.

In [3]:
from model import load_box, box_input
box = load_box()
for state, action in [(71, 36), (11, 83)]:
    p = F.softmax(box(box_input(torch.tensor([state]), F.one_hot(torch.tensor([action]), 2 * d).float())), -1)[0]
    print(f'{state} + {action}: box says {p.argmax().item()} with probability {p.max():.3f}   (true {(state + action) % d})')

71 + 36: box says 11 with probability 0.996   (true 11)
11 + 83: box says 94 with probability 0.996   (true 94)


## What the box does with a muddle

Before training, attention is spread evenly, so the box receives a **blend** of actions. It does not answer "one of these three"; most of its probability goes elsewhere. This is why long chains start near zero success.

In [4]:
state, actions = 71, [83, 24, 36]
blend = sum(F.one_hot(torch.tensor([a]), 2 * d).float() for a in actions) / 3
p = F.softmax(box(box_input(torch.tensor([state]), blend)), -1)[0]
for a in actions:
    print(f'via +{a}: probability of {(state + a) % d} is {p[(state + a) % d]:.3f}')
print(f'probability on the other {d - 3} states: {1 - sum(p[(state + a) % d] for a in actions):.3f}')

via +83: probability of 58 is 0.043
via +24: probability of 95 is 0.111
via +36: probability of 11 is 0.196
probability on the other 93 states: 0.650
